# Vector search from scratch: hands-on (optional)
**LLMs, Animated · Lesson SY.2 Embeddings & vector search**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

You'll build, with NumPy only and nothing to download:
1. cosine similarity and row normalization,
2. brute-force top-k search,
3. a tiny BM25 keyword scorer,
4. reciprocal rank fusion (hybrid search),
5. an IVF-style approximate index, measured by recall@k against brute force.

The embedding model is a **toy** defined in the setup cell (about 60 known words plus hashed words), so you can see every number. Solutions are at the end.

In [ ]:

# --- A toy embedding model (no downloads) ---------------------------------
# 6 hand-made "topic" directions (billing, refunds, account, shipping, app problems,
# privacy) from a small word list, plus a hashed bag of words. Real embedding models
# learn hundreds of directions from data; this one is just big enough to show the ideas.
import re, zlib
import numpy as np

TOPICS = ["billing", "refunds", "account", "shipping", "app", "privacy"]
LEX = {
    "card": [1,0,0,0,0,0], "pay": [1,0,0,0,0,0], "payment": [1,0,0,0,0,0], "charged": [1,.2,0,0,0,0],
    "charge": [1,.2,0,0,0,0], "billing": [1,0,0,0,0,0], "invoice": [1,0,0,0,0,0], "declined": [.9,0,0,0,0,0],
    "price": [.8,0,0,0,0,0], "subscription": [.9,0,.2,0,0,0], "yearly": [.8,0,0,0,0,0],
    "money": [.5,.6,0,0,0,0], "refund": [.2,1,0,0,0,0], "refunds": [.2,1,0,0,0,0], "back": [0,.4,0,0,0,0],
    "return": [0,.9,0,.3,0,0], "returned": [0,.9,0,.3,0,0], "exchange": [0,.8,0,.2,0,0], "credit": [.4,.6,0,0,0,0],
    "login": [0,0,1,0,0,0], "log": [0,0,.9,0,0,0], "sign": [0,0,.9,0,0,0], "password": [0,0,1,0,0,.1],
    "account": [0,0,.8,0,0,.2], "email": [0,0,.8,0,0,.1], "locked": [0,0,1,0,0,0], "two": [0,0,.3,0,0,.1],
    "parcel": [0,0,0,1,0,0], "package": [0,0,0,1,0,0], "delivery": [0,0,0,1,0,0], "delivered": [0,0,0,1,0,0],
    "shipping": [0,0,0,1,0,0], "track": [0,0,0,1,0,0], "order": [.2,.1,0,.8,0,0], "address": [0,0,.2,.8,0,0],
    "late": [0,0,0,.8,0,0], "where": [0,0,0,.4,0,0], "country": [0,0,0,.6,0,0],
    "battery": [0,0,0,0,1,0], "crash": [0,0,0,0,1,0], "crashes": [0,0,0,0,1,0], "slow": [0,0,0,0,1,0],
    "error": [.2,0,.1,0,.7,0], "update": [0,0,0,0,.7,0], "app": [0,0,0,0,.6,0], "sync": [0,0,0,0,.9,0],
    "data": [0,0,.1,0,0,1], "privacy": [0,0,0,0,0,1], "delete": [0,0,.4,0,0,.8], "download": [0,0,0,0,.2,.7],
    "personal": [0,0,0,0,0,.9], "export": [0,0,0,0,0,.9],
}
STOP = set("a an the my me i is are was to of for and or on in with your you it this that can't cant can not "
           "how do does what why be by at from if any all".split())
H, WB = 48, 0.5

def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower().replace("can't", "cant"))

def _stem(t):
    for suf in ("ing", "es", "s", "ed"):
        if len(t) > 4 and t.endswith(suf):
            return t[: -len(suf)]
    return t

def embed(text):
    """Return a unit-length vector of size 6 + 48 for a piece of text."""
    topic = np.zeros(6)
    for t in tokens(text):
        if t in LEX:
            topic += LEX[t]
    if topic.any():
        topic /= np.linalg.norm(topic)
    words = np.zeros(H)
    for t in tokens(text):
        if t in STOP or any(c.isdigit() for c in t):   # codes like E1042 carry no "meaning"
            continue
        words[zlib.crc32(_stem(t).encode()) % H] += 1
    if words.any():
        words /= np.linalg.norm(words)
    v = np.concatenate([topic, WB * words])
    n = np.linalg.norm(v)
    return v / n if n else v

DOCS = [
    "Update the card you pay with",
    "Error E1042: your card was declined",
    "Why was I charged twice this month?",
    "Switch to yearly billing and save",
    "Request a refund within 30 days",
    "Return an item for a refund",
    "Refunds reach your bank in 5 to 10 days",
    "Reset a forgotten password",
    "Turn on two-step sign-in",
    "Locked out after too many attempts",
    "Change the email on your account",
    "Track your parcel with the order ID",
    "Delivery times by country",
    "Parcel says delivered but isn't here",
    "Change the delivery address",
    "The app drains my battery",
    "Fix crashes after the latest update",
    "Photo sync is slow on Wi-Fi",
    "Download a copy of your data",
    "Delete your account and all data",
]
E = np.stack([embed(d) for d in DOCS])   # one row per document
print(E.shape, "document matrix ready")


## 1 · Cosine similarity
Write `cosine(a, b)` for two 1-D vectors, and `normalize_rows(M)` that scales every row of a matrix to length 1 (leave all-zero rows as zeros).

$$\cos(a,b) = \frac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}$$

In [ ]:
def cosine(a, b):
    # TODO: dot product divided by both lengths
    raise NotImplementedError

def normalize_rows(M):
    # TODO: divide each row by its length; keep zero rows at zero
    raise NotImplementedError


In [ ]:
assert abs(cosine(np.array([1, 2, 2]), np.array([2, 1, 2])) - 8 / 9) < 1e-9
assert abs(cosine(np.array([1, 2, 2]), np.array([0, 3, 4])) - 14 / 15) < 1e-9
N = normalize_rows([[3, 4], [0, 0], [1, 0]])
assert np.allclose(N, [[0.6, 0.8], [0, 0], [1, 0]])
print("✓ cosine and normalize_rows work")

## 2 · Brute-force top-k
`E` already has unit-length rows, so cosine is a dot product. Write `top_k(query, k)`: embed the query, score every document, and return the indices of the `k` best, best first.

In [ ]:
def top_k(query, k=3):
    # TODO: q = embed(query); scores = E @ q; return the k best indices, highest score first
    raise NotImplementedError


In [ ]:
r = top_k("get my money back", 3)
assert set(r) == {4, 5, 6}, [DOCS[i] for i in r]
assert top_k("delete my account data", 1) == [19]   # "delete" and "account" both point at doc 19
assert 19 in top_k("delete my data", 2)
print("✓ top_k:", [DOCS[i] for i in r])

## 3 · A tiny BM25
BM25 scores a document $d$ for query words $t$:

$$\text{BM25}(d) = \sum_{t} \text{idf}(t)\,\frac{f_{t,d}\,(k_1+1)}{f_{t,d} + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}, \qquad \text{idf}(t) = \ln\!\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

$f_{t,d}$ is how often $t$ occurs in $d$, $|d|$ the document length in words, avgdl the average length, $N$ the number of documents and $n_t$ how many contain $t$. Use $k_1 = 1.2$, $b = 0.75$, lower-cased `tokens(...)`, and drop words in `STOP`.

In [ ]:
DOC_TOKS = [[t for t in tokens(d) if t not in STOP] for d in DOCS]

def bm25_scores(query, k1=1.2, b=0.75):
    """Return a numpy array with one BM25 score per document."""
    # TODO
    raise NotImplementedError


In [ ]:
s = bm25_scores("error E1042")
assert int(np.argmax(s)) == 1 and (s > 0).sum() == 1
assert bm25_scores("get my money back").sum() == 0   # no shared words at all
assert abs(bm25_scores("e1042")[1] - bm25_scores("E1042")[1]) < 1e-12
print("✓ BM25 finds the exact code; dense top-3 for the same query:", [DOCS[i] for i in top_k("error E1042")])

## 4 · Reciprocal rank fusion
Merge ranked lists by rank, not score: $\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}$ with ranks starting at 1 and $k = 60$. Write `rrf(lists, k=60)` that returns `(doc, score)` pairs sorted best first. Then `hybrid(query, n)` should fuse the full dense ranking with the BM25 ranking (only documents with a positive BM25 score) and return the top `n` indices.

In [ ]:
def rrf(lists, k=60):
    # TODO: lists is a list of ranked lists of ids
    raise NotImplementedError

def hybrid(query, n=3):
    # TODO: dense = top_k(query, len(DOCS)); keyword = ids with BM25 > 0, best first
    raise NotImplementedError


In [ ]:
fused = rrf([["D3", "D1", "D7", "D2"], ["D7", "D3", "D5"]])
assert [d for d, _ in fused] == ["D3", "D7", "D1", "D5", "D2"]
assert abs(fused[0][1] - (1/61 + 1/62)) < 1e-12
assert hybrid("error E1042")[0] == 1
print("✓ hybrid top-3 for 'error E1042':", [DOCS[i] for i in hybrid("error E1042")])

## 5 · An approximate index and its recall@k
Twenty documents don't need an index, so we switch to 20,000 random 32-dimensional unit vectors in 50 loose clusters. The setup below builds an **IVF** index for you: k-means-style centroids and, for every vector, the list it belongs to.

Write `ivf_search(q, k, nprobe)`: compare `q` with the centroids, scan only the vectors in the `nprobe` closest lists, and return the `k` best ids. Then `recall_at_k(approx, exact)` = fraction of the exact top-k that the approximate search found.

In [ ]:
rng = np.random.default_rng(0)
D, NV, NC = 32, 20_000, 50
centres = normalize_rows(rng.normal(size=(NC, D)))
X = normalize_rows(centres[rng.integers(0, NC, NV)] + 0.35 * rng.normal(size=(NV, D)))
# a few rounds of k-means (dot product on unit vectors) to build the inverted lists
C = X[rng.choice(NV, 64, replace=False)]
for _ in range(5):
    assign = np.argmax(X @ C.T, axis=1)
    C = normalize_rows(np.stack([X[assign == j].mean(0) if (assign == j).any() else C[j] for j in range(len(C))]))
assign = np.argmax(X @ C.T, axis=1)
LISTS = [np.where(assign == j)[0] for j in range(len(C))]

def exact_search(q, k):
    return np.argsort(-(X @ q))[:k]

def ivf_search(q, k=10, nprobe=4):
    # TODO: pick the nprobe centroids with the highest dot product with q,
    # gather the ids in those LISTS, score only them, return the k best ids
    raise NotImplementedError

def recall_at_k(approx, exact):
    # TODO
    raise NotImplementedError


In [ ]:
assert recall_at_k([1, 2, 3], [3, 2, 9]) == 2 / 3
Q = normalize_rows(X[rng.choice(NV, 100, replace=False)] + 0.2 * rng.normal(size=(100, D)))
rec = {p: np.mean([recall_at_k(ivf_search(q, 10, p), exact_search(q, 10)) for q in Q]) for p in (1, 4, 16, 64)}
assert rec[64] == 1.0                      # probing every list is exact search
assert rec[1] <= rec[4] <= rec[16] <= rec[64]
print("✓ recall@10 by nprobe:", {p: round(r, 3) for p, r in rec.items()})

## Experiment
How much of the collection does each `nprobe` scan, and what recall does it buy? Plot recall@10 against the fraction of vectors scanned. Then try changing the noise level `0.35` in exercise 5 (more noise means blurrier clusters) and re-run.

In [ ]:
import matplotlib.pyplot as plt
probes = [1, 2, 4, 8, 16, 32]
frac, recs = [], []
for p in probes:
    scanned, r = [], []
    for q in Q:
        best_lists = np.argsort(-(C @ q))[:p]
        scanned.append(sum(len(LISTS[j]) for j in best_lists) / NV)
        r.append(recall_at_k(ivf_search(q, 10, p), exact_search(q, 10)))
    frac.append(np.mean(scanned)); recs.append(np.mean(r))
plt.plot(frac, recs, "o-")
for p, x, y in zip(probes, frac, recs):
    plt.annotate(f"nprobe={p}", (x, y), textcoords="offset points", xytext=(6, -12), fontsize=8)
plt.xlabel("fraction of vectors scanned"); plt.ylabel("recall@10"); plt.title("IVF: recall vs work")
plt.grid(alpha=.3); plt.show()

## Solutions
Try the exercises first. Each solution below can be pasted into its exercise cell.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def normalize_rows(M):
    M = np.asarray(M, dtype=float)
    n = np.linalg.norm(M, axis=1, keepdims=True)
    return np.divide(M, n, out=np.zeros_like(M), where=n > 0)


### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def top_k(query, k=3):
    q = embed(query)
    scores = E @ q
    return [int(i) for i in np.argsort(-scores)[:k]]


### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
DOC_TOKS = [[t for t in tokens(d) if t not in STOP] for d in DOCS]

def bm25_scores(query, k1=1.2, b=0.75):
    N = len(DOC_TOKS)
    avgdl = sum(len(d) for d in DOC_TOKS) / N
    q = [t for t in tokens(query) if t not in STOP]
    scores = np.zeros(N)
    for t in q:
        n_t = sum(1 for d in DOC_TOKS if t in d)
        if n_t == 0:
            continue
        idf = np.log(1 + (N - n_t + 0.5) / (n_t + 0.5))
        for i, d in enumerate(DOC_TOKS):
            f = d.count(t)
            scores[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
    return scores


### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def rrf(lists, k=60):
    scores = {}
    for lst in lists:
        for rank, d in enumerate(lst, start=1):
            scores[d] = scores.get(d, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: -kv[1])

def hybrid(query, n=3):
    dense = top_k(query, len(DOCS))
    s = bm25_scores(query)
    keyword = [int(i) for i in np.argsort(-s) if s[i] > 0]
    return [d for d, _ in rrf([dense, keyword])[:n]]


### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
rng = np.random.default_rng(0)
D, NV, NC = 32, 20_000, 50
centres = normalize_rows(rng.normal(size=(NC, D)))
X = normalize_rows(centres[rng.integers(0, NC, NV)] + 0.35 * rng.normal(size=(NV, D)))
C = X[rng.choice(NV, 64, replace=False)]
for _ in range(5):
    assign = np.argmax(X @ C.T, axis=1)
    C = normalize_rows(np.stack([X[assign == j].mean(0) if (assign == j).any() else C[j] for j in range(len(C))]))
assign = np.argmax(X @ C.T, axis=1)
LISTS = [np.where(assign == j)[0] for j in range(len(C))]

def exact_search(q, k):
    return np.argsort(-(X @ q))[:k]

def ivf_search(q, k=10, nprobe=4):
    best_lists = np.argsort(-(C @ q))[:nprobe]
    ids = np.concatenate([LISTS[j] for j in best_lists])
    scores = X[ids] @ q
    return ids[np.argsort(-scores)[:k]]

def recall_at_k(approx, exact):
    return len(set(map(int, approx)) & set(map(int, exact))) / len(exact)
